# Pooling methods — global, local, and Type C (numeric eval)

Run in the repo root (local `crowd-gpu` or Colab/Kaggle after `pip install -e .`).

Encodes gallery clips **once**, then scores with:

| Pool | Meaning |
|------|--------|
| `mean` | Global average of clip embeddings |
| `query_max` | Best clip–text cosine (local) |
| `global_local` | `α·global + (1−α)·local` (Method C) |

One code cell prints metrics per method; the last cell draws bar charts. Change `MODEL` / `DATASET_KEY` to compare encoders.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd()
if not (REPO / "shawaf_vlm").is_dir():
    REPO = Path.cwd().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

MODEL = "pe_core_l14"  # e.g. irra, siglip2, qwen3_vl_embed_2b
DATASET_KEY = "tvpreid:prid"  # tvpreid:ilids, tvpreid:duke, rstpreid:test, groot:combined
SPLIT = "test"
DEVICE = "cuda"
BATCH = 4
TEXT_BATCH = 16
HYBRID_ALPHA = 0.5

WINDOW = {
    "name": "reid_8fps_n64",
    "num_frames": 8,
    "stride": 4,
    "sample_fps": 8.0,
    "max_frames": 64,
}
TYPE_C_POOLS = ("mean", "query_max", "global_local")
FRAME_CACHE = REPO / ".cache" / "frame_cache"
RESULTS_DIR = REPO / "results" / "pooling_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(REPO, MODEL, DATASET_KEY)

In [ ]:
import json
from datetime import datetime, timezone

import torch
from shawaf_vlm.eval_loop import evaluate_text_to_tracklet_windows
from shawaf_vlm.metrics import format_metrics
from shawaf_vlm.models.registry import build_encoder
from shawaf_vlm.retrieval.datasets import load_splits

splits = load_splits(DATASET_KEY, split=SPLIT, download=False)
encoder = build_encoder(MODEL, device=DEVICE)
print("Gallery", len(splits.gallery), "queries", len(splits.query))

## Encode once (all pooling modes share the same clip features)

In [ ]:
scored = evaluate_text_to_tracklet_windows(
    encoder,
    splits,
    num_frames=WINDOW["num_frames"],
    stride=WINDOW["stride"],
    sample_fps=WINDOW["sample_fps"],
    max_frames=WINDOW["max_frames"],
    pools=TYPE_C_POOLS,
    batch_size=BATCH,
    text_batch_size=TEXT_BATCH,
    junk_same_camera=False,
    frame_cache=FRAME_CACHE,
    hybrid_alpha=HYBRID_ALPHA,
)
payload = {
    "model": MODEL,
    "dataset": DATASET_KEY,
    "protocol": WINDOW["name"],
    "hybrid_alpha": HYBRID_ALPHA,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "metrics": {pool: {k: float(v) for k, v in m.items()} for pool, m in scored.items()},
}
out_json = RESULTS_DIR / f"{MODEL}_{DATASET_KEY.replace(':', '_')}_{WINDOW['name']}_type_c.json"
out_json.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote", out_json)

### Method: global mean (`mean`)

In [ ]:
print(format_metrics(scored["mean"]))

### Method: local max (`query_max`)

In [ ]:
print(format_metrics(scored["query_max"]))

### Method: Type C hybrid (`global_local`)

In [ ]:
print(f"α={HYBRID_ALPHA} (global), {1 - HYBRID_ALPHA:.2f} (local)")
print(format_metrics(scored["global_local"]))

## Bar charts (Rank-1 and mAP)

In [ ]:
import matplotlib.pyplot as plt

labels = {
    "mean": "Global mean",
    "query_max": "Local max",
    "global_local": f"Type C (α={HYBRID_ALPHA})",
}
r1 = [scored[p]["Rank-1"] for p in TYPE_C_POOLS]
mapv = [scored[p]["mAP"] for p in TYPE_C_POOLS]
xt = [labels[p] for p in TYPE_C_POOLS]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].bar(xt, r1, color=["#4c78a8", "#f58518", "#54a24b"])
axes[0].set_title("Rank-1 (%)")
axes[0].set_ylim(0, max(5, max(r1) * 1.15))
axes[1].bar(xt, mapv, color=["#4c78a8", "#f58518", "#54a24b"])
axes[1].set_title("mAP (%)")
axes[1].set_ylim(0, max(5, max(mapv) * 1.15))
for ax in axes:
    ax.tick_params(axis="x", rotation=12)
fig.suptitle(f"{MODEL} | {DATASET_KEY} | {WINDOW['name']}", y=1.02)
plt.tight_layout()
png = RESULTS_DIR / f"{MODEL}_{DATASET_KEY.replace(':', '_')}_{WINDOW['name']}_bars.png"
fig.savefig(png, dpi=150, bbox_inches="tight")
print("Saved", png)
plt.show()

del encoder
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Optional: compare multiple models (load saved JSON files)

In [ ]:
import pandas as pd

rows = []
for path in sorted(RESULTS_DIR.glob("*_type_c.json")):
    blob = json.loads(path.read_text(encoding="utf-8"))
    for pool, metrics in blob["metrics"].items():
        rows.append(
            {
                "model": blob["model"],
                "dataset": blob["dataset"],
                "pool": pool,
                "Rank-1": metrics["Rank-1"],
                "mAP": metrics["mAP"],
            }
        )
if not rows:
    print("No saved runs in", RESULTS_DIR)
else:
    df = pd.DataFrame(rows)
    display(df.pivot_table(index=["model", "dataset"], columns="pool", values="Rank-1").round(2))
    for (model, dataset), part in df.groupby(["model", "dataset"]):
        part = part.set_index("pool").reindex(TYPE_C_POOLS)
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.bar([labels[p] for p in TYPE_C_POOLS], part["Rank-1"], color=["#4c78a8", "#f58518", "#54a24b"])
        ax.set_title(f"{model} | {dataset} | Rank-1")
        plt.tight_layout()
        plt.show()